# Part 3 — Machine Learning: Predict No-Shows

CareLoop Clinics — No-Show Guard

Load `appointments_clean.csv`, build features, split, train models, choose honestly, output `test_risk_scores.csv` and `results_part3.json`.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings("ignore")

print("Imports loaded.")

## Task 3.1 — Features, Split, Baseline

In [ ]:
# Load cleaned data (do NOT shuffle or re-sort)
df = pd.read_csv("appointments_clean.csv")
print(f"Loaded: {df.shape}")

# Target
y = df["no_show"]

# Features
X = df.copy()

# reminder_sent: Yes/No -> 1/0
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encode booking_channel and visit_type
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], drop_first=False)

# Drop excluded columns (keep lead_days!)
drop_cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
             "gender", "status", "no_show"]
X = X.drop(columns=[c for c in drop_cols if c in X.columns])

print(f"X shape: {X.shape}")
print(f"X columns ({len(X.columns)}): {list(X.columns)}")
print(f"y shape: {y.shape}")
print(f"No-show rate overall: {y.mean()*100:.1f}%")

In [ ]:
# Split (test_size=0.2, random_state=42, no stratify, no shuffle after load)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Test no-shows: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

# Baseline: always predict 0 (Attended)
baseline_acc = (y_test == 0).mean()
print(f"Baseline accuracy (always Attended): {baseline_acc:.4f} ({int((y_test==0).sum())}/{len(y_test)})")

# Keep test appointment_ids for later
test_appt_ids = df.loc[X_test.index, "appointment_id"].values

## Task 3.2 — Logistic Regression

In [ ]:
# Scale features (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# Predict on test
y_pred = lr.predict(X_test_s)
y_proba = lr.predict_proba(X_test_s)[:, 1]

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

# Coefficients
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_[0]
}).sort_values("coefficient", ascending=False)
print("\nCoefficients (sorted):")
print(coef_df.to_string(index=False))

In [ ]:
# np.exp of reminder_sent coefficient
rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr.coef_[0][rem_idx]
exp_coef = np.exp(rem_coef)
print(f"\nreminder_sent coefficient: {rem_coef:.4f}")
print(f"np.exp(coef): {exp_coef:.4f}")

# Real No->Yes effect: divide by feature's std (scaler.scale_)
rem_scale = scaler.scale_[rem_idx]
real_effect = np.exp(rem_coef / rem_scale)
print(f"reminder_sent scale (std): {rem_scale:.4f}")
print(f"np.exp(coef/scale): {real_effect:.4f}")
print(f"\nInterpretation: In this model, holding other features constant, ")
print(f"the odds of a no-show are about {(1-real_effect)*100:.0f}% lower ")
print(f"for appointments with a reminder sent vs not sent — an association, not proof of cause.")
print(f"\nWhy they differ: np.exp(coef) gives odds ratio per 1 SD change in the scaled feature, ")
print(f"while np.exp(coef/scale) gives odds ratio for the actual 0->1 change in the original binary feature.")

## Task 3.3 — Decision Tree and Overfitting

In [ ]:
# Full depth tree (no limit)
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

train_acc_full = dt_full.score(X_train, y_train)
test_acc_full = dt_full.score(X_test, y_test)

print(f"Full-depth tree:")
print(f"  Training accuracy: {train_acc_full:.4f}")
print(f"  Test accuracy: {test_acc_full:.4f}")
print(f"  Tree depth: {dt_full.get_depth()}")
print(f"  Gap shows: The model memorized noise (overfitting), ")
print(f"  performing below the {baseline_acc:.4f} baseline on test data.")

# Depth-limited tree (max_depth=3)
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)

train_acc_3 = dt_3.score(X_train, y_train)
test_acc_3 = dt_3.score(X_test, y_test)

print(f"\nDepth-3 tree:")
print(f"  Training accuracy: {train_acc_3:.4f}")
print(f"  Test accuracy: {test_acc_3:.4f}")
print(f"  Gap shows: Much less overfitting; test accuracy close to training, ")
print(f"  and above the baseline.")

## Task 3.4 — Random Forest and Comparison Table

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print(f"Random Forest:")
print(f"  Test Accuracy: {acc_rf:.4f}")
print(f"  Test Recall: {rec_rf:.4f}")
print(f"  Test ROC-AUC: {auc_rf:.4f}")

# Feature importances (top 3)
imp_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print(f"\nTop 3 feature importances:")
print(imp_df.head(3).to_string(index=False))

In [ ]:
# Comparison table
print("\nComparison Table (Test Set):")
print(f"{'Model':<25} {'Accuracy':>10} {'Recall':>10} {'ROC-AUC':>10}")
print(f"{'-'*25} {'-'*10} {'-'*10} {'-'*10}")
print(f"{'Logistic Regression':<25} {acc:>10.4f} {rec:>10.4f} {auc:>10.4f}")
dt3_pred = dt_3.predict(X_test)
dt3_proba = dt_3.predict_proba(X_test)[:, 1]
print(f"{'Decision Tree (depth=3)':<25} {test_acc_3:>10.4f} {recall_score(y_test, dt3_pred):>10.4f} {roc_auc_score(y_test, dt3_proba):>10.4f}")
print(f"{'Random Forest':<25} {acc_rf:>10.4f} {rec_rf:>10.4f} {auc_rf:>10.4f}")

# Which looks best?
print(f"\nLogistic Regression has the highest test ROC-AUC ({auc:.4f}) ")
print(f"of the three models. It also has better precision-recall balance.")
print(f"This is for reporting; formal choice is made in Task 3.5 from training data only.")

## Task 3.5 — Tune, Choose Model & Threshold, Test Once

In [ ]:
# (a) GridSearchCV on Random Forest
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
gs.fit(X_train, y_train)

print(f"Best params: {gs.best_params_}")
print(f"Best CV ROC-AUC (best_score_): {gs.best_score_:.4f}")

# Best estimator
rf_tuned = gs.best_estimator_

In [ ]:
# (b) Choose final model using CV on training data only

# Logistic Regression with pipeline (scaler refit each fold)
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
lr_cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

# Depth-3 Tree
dt3_cv_auc = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                              X_train, y_train, cv=5, scoring="roc_auc").mean()

# Random Forest (default)
rf_cv_auc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                             X_train, y_train, cv=5, scoring="roc_auc").mean()

# Tuned Random Forest
rf_tuned_cv_auc = gs.best_score_

print(f"5-fold CV ROC-AUC on training set:")
print(f"  Logistic Regression (pipeline): {lr_cv_auc:.4f}")
print(f"  Depth-3 Tree: {dt3_cv_auc:.4f}")
print(f"  Random Forest (default): {rf_cv_auc:.4f}")
print(f"  Tuned Random Forest: {rf_tuned_cv_auc:.4f}")

# Choose winner (highest CV AUC)
cv_scores = {
    "Logistic Regression": lr_cv_auc,
    "Depth-3 Tree": dt3_cv_auc,
    "Random Forest": rf_cv_auc,
    "Tuned Random Forest": rf_tuned_cv_auc
}
winner_name = max(cv_scores, key=cv_scores.get)
winner_auc = cv_scores[winner_name]
print(f"\nWinner: {winner_name} (CV AUC = {winner_auc:.4f})")

In [ ]:
# (c) Choose threshold using out-of-fold probabilities on training data

if winner_name == "Logistic Regression":
    # Use pipeline for OOF predictions
    oof_proba = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
    final_model = lr_pipe
elif winner_name == "Depth-3 Tree":
    oof_proba = cross_val_predict(DecisionTreeClassifier(max_depth=3, random_state=42),
                                   X_train, y_train, cv=5, method="predict_proba")[:, 1]
    final_model = DecisionTreeClassifier(max_depth=3, random_state=42)
elif winner_name == "Random Forest":
    oof_proba = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=42),
                                   X_train, y_train, cv=5, method="predict_proba")[:, 1]
    final_model = RandomForestClassifier(n_estimators=100, random_state=42)
else:  # Tuned Random Forest
    oof_proba = cross_val_predict(rf_tuned, X_train, y_train, cv=5, method="predict_proba")[:, 1]
    final_model = rf_tuned

# Evaluate thresholds on OOF
for thresh in [0.5, 0.35]:
    oof_pred = (oof_proba >= thresh).astype(int)
    prec = precision_score(y_train, oof_pred)
    rec = recall_score(y_train, oof_pred)
    f1 = f1_score(y_train, oof_pred)
    flagged = oof_pred.sum()
    print(f"Threshold {thresh}: precision={prec:.4f}, recall={rec:.4f}, F1={f1:.4f}, flagged={flagged}")

# Choose threshold with higher F1
thresh_05_f1 = f1_score(y_train, (oof_proba >= 0.5).astype(int))
thresh_35_f1 = f1_score(y_train, (oof_proba >= 0.35).astype(int))
chosen_thresh = 0.35 if thresh_35_f1 > thresh_05_f1 else 0.5
print(f"\nChosen threshold: {chosen_thresh} (higher F1)")
print(f"Why recall matters: A missed warning costs an empty doctor slot; ")
print(f"an extra follow-up call costs about a minute of staff time.")

In [ ]:
# (d) Fit winner on full training set, evaluate once on test at chosen threshold
final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= chosen_thresh).astype(int)

test_prec = precision_score(y_test, test_pred)
test_rec = recall_score(y_test, test_pred)
test_flagged = test_pred.sum()
test_auc = roc_auc_score(y_test, test_proba)

print(f"Final Test Evaluation (threshold={chosen_thresh}):")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall: {test_rec:.4f}")
print(f"  Flagged: {test_flagged} out of {len(y_test)}")
print(f"  ROC-AUC: {test_auc:.4f}")

# (e) Save test_risk_scores.csv
risk_df = pd.DataFrame({
    "appointment_id": test_appt_ids,
    "no_show": y_test.values,
    "risk_prob": test_proba,
    "flag": test_pred
})
risk_df.to_csv("test_risk_scores.csv", index=False)
print(f"\nSaved test_risk_scores.csv: {risk_df.shape}")
print(f"Flag count in file: {risk_df['flag'].sum()}")

In [ ]:
# Save results to results_part3.json for README
top_coefs = coef_df.head(5).to_dict(orient="records")
results = {
    "winner": winner_name,
    "cv_auc": round(float(winner_auc), 4),
    "threshold": float(chosen_thresh),
    "test_precision": round(float(test_prec), 4),
    "test_recall": round(float(test_rec), 4),
    "test_flagged": int(test_flagged),
    "test_auc": round(float(test_auc), 4),
    "top_coefficients": top_coefs,
    "lr_cv_auc": round(float(lr_cv_auc), 4),
    "dt3_cv_auc": round(float(dt3_cv_auc), 4),
    "rf_cv_auc": round(float(rf_cv_auc), 4),
    "rf_tuned_cv_auc": round(float(rf_tuned_cv_auc), 4),
    "baseline_accuracy": round(float(baseline_acc), 4),
    "test_no_shows": int(y_test.sum()),
    "train_size": int(X_train.shape[0]),
    "test_size": int(X_test.shape[0]),
    "n_features": int(X.shape[1])
}
with open("results_part3.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved results_part3.json")
print(json.dumps(results, indent=2))

In [ ]:
# Real-world threshold selection
print("\nHow a clinic would really choose the threshold:")
print("By how many follow-up calls the front desk can make in an evening,")
print("against the cost of an empty slot.")